# 任務 2：檢查資料品質


## 開始之前
姓名：歐國澤 學號：C114118111
###操作方式
1. 先執行下一個完整的資料準備儲存格。每份任務可獨立執行；重啟環境後請由上而下重新執行。
2. 在每個小節的程式中，將 `___` 替換成符合題意的程式碼，並把 `# 學生註解` 改成自己的繁體中文解說（至少說明指定程式的輸入／處理／輸出）。`___` 是刻意留下的空格，完成前執行會報錯。
3. 先記錄預測，執行後完成驗證與解釋；再完成延伸練習。遇到困難可以使用小節專屬 AI 提示詞，仍須檢查、修改、執行與理解。
4. 在文字儲存格中填寫答案，保留程式、執行結果、圖表和 AI 使用紀錄。依教師指示交付已完成的 `.ipynb`。首次執行任務 1 的存檔程式會覆寫同名 practice_student_data.csv，請先確認或改檔名。


In [ ]:
# 載入分析與繪圖套件；np、pd、plt 是慣用縮寫
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 固定種子及呼叫順序，使本課範例能重現
rng = np.random.default_rng(42)
n = 60  # 模擬學生人數
departments = ["資訊管理", "企業管理", "金融管理"]
# 字典每個鍵是欄位名稱，每個值提供 60 位學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)  # 把字典轉成資料表
noise = rng.normal(0, 5, n)  # 模擬其他未觀察的差異
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)  # 限制 0 到 100 分
df.loc[[5, 17], "sleep_hours"] = np.nan  # 成績算完後才加入缺失
df.loc[8, "screen_time_hours"] = 15.0  # 成績算完後才加入異常值
print("資料已準備：", df.shape)

## 2.1 缺失值
`isna()` 將每個值轉為是否缺失的布林值；`sum()` 統計 True 的數量。`NaN` 不等於零，也不是已知的睡眠 0 小時。
**預測：** 哪個欄位缺少幾筆？

### 學生練習 2.1：完成程式與註解
**我的預測：**`sleep_hours` 欄位會少 2 筆，其他欄位都是 0 筆缺失。

**AI求助提示詞：**
> 我是 Python 初學者，正在完成資料分析任務 2.1。我的資料表叫 df，請教我如何補上逐欄統計缺失值數量的運算。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 輸入資料表 df，先用 isna() 判斷缺失再用 sum() 加總，輸出每欄缺失筆數給 missing_count
missing_count = df.isna().sum()
print(missing_count)
missing_condition = df["sleep_hours"].isna()
missing_rows = df[missing_condition]
print(missing_rows[["student_id", "sleep_hours", "quiz_score"]])
# df[missing_condition] 用布林條件把睡眠時間缺漏的學生那一列篩選出來


### 驗證、解釋與延伸 2.1
**延伸練習的預測與驗證**預測 `quiz_score` 缺失數是 0，實際跑出來是 `0`，跟上面 `missing_count` 的結果一致。
**AI求助提示詞：**
> 請引導我完成這個延伸練習：統計 quiz_score 的缺失數並核對整欄統計結果。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 挑出 quiz_score 這一欄，用 isna().sum() 計算並印出缺失筆數
print("quiz_score 缺失數：", df["quiz_score"].isna().sum())


## 2.2 合理範圍與異常
本資料生成規格規定螢幕時間原先在 1～10 小時，之後故意設為 15。真實資料的極端值應查證，不應僅憑數值直接刪除。
**預測：** 超過 10 小時的是誰？

### 學生練習2.2：完成程式與註解
**我的預測**會找出 1 筆螢幕時間 15.0 小時的資料，就是索引 8 的那位學生（學號 9）。
**AI求助提示詞：**
> 我是 Python 初學者，正在完成資料分析任務 2.2。我的資料表叫 df，請教我如何補上篩選螢幕時間超過 10 小時的布林條件。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 輸入 screen_time_hours 欄位與門檻 10，判斷是否大於 10，輸出布林結果給 over_limit
over_limit = df["screen_time_hours"] > 10
print(df.loc[over_limit, ["student_id", "screen_time_hours"]])
print(df["screen_time_hours"].describe())
# describe() 用來列出該欄位的筆數、平均、標準差與最大最小值等摘要


###驗證、解釋與延伸 2.2
**延伸練習的預測與驗證**預測改門檻 8 後會篩出更多筆資料（包含原本 15.0 的那位），實際執行後印出所有大於 8 的學生編號與總筆數。

**AI求助提示詞：**
> 請引導我完成這個延伸練習：把門檻改為 8，列出符合條件的編號與數量。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 設定門檻大於 8 篩選資料，印出符合的 student_id 與總數量
over_8 = df["screen_time_hours"] > 8
print(df.loc[over_8, ["student_id", "screen_time_hours"]])
print("超過 8 小時人數：", over_8.sum())


## 2.3 重複值與紀錄
完整資料列重複和編號重複是不同問題。先紀錄，再決定是否處理。

### 學生練習 2.3：完成程式與註解

**我的預測**學號是從 1 到 60 依序產生的，預期完整列重複與編號重複都是 0 筆。

**AI求助提示詞：**
> 我是 Python 初學者，正在完成資料分析任務 2.3。我的資料表叫 df，請教我如何補上只依 student_id 統計重複筆數的運算。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [ ]:
# 輸入指定欄位 subset=["student_id"]，用 duplicated() 找出重複並以 sum() 加總輸出筆數
print("完整列重複：", df.duplicated().sum())
print("編號重複：", df.duplicated(subset=["student_id"]).sum())
# df.duplicated().sum() 用來檢查整列所有欄位都完全一模一樣的重複資料有幾筆


###驗證、解釋與延伸 2.3
**延伸練習的預測與驗證**預測每個學號都只會出現 1 次，實際用 `value_counts()` 查看前幾筆確實都是 1 次。

**AI求助提示詞：**
> 請引導我完成這個延伸練習：用 value_counts 檢查 student_id 的出現次數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [ ]:
# 輸入 student_id 欄位，用 value_counts() 統計各編號出現次數並印出前 5 筆
print(df["student_id"].value_counts().head())


## 任務交付與 AI 使用紀錄
- 使用的 AI 提示詞：用講義提供的 2.1～2.3 提示詞查 `isna().sum()`、布林篩選條件跟 `duplicated(subset=...)` 的用法。
- AI 建議中我修改或拒絕的一處及原因：把 AI 原本較長較制式的說明精簡掉，改寫成口語、只留輸入／處理／輸出重點的短註解。
- 遇到的錯誤或疑問、修正與驗證方式：一開始疑問 `student_id` 為什麼是 6 和 18 而不是 5 和 17，後來對照準備儲存格才確認是因為編號從 1 開始、索引從 0 開始，核對 `df.loc[[5, 17]]` 後無誤。
- 本任務一項有數據支持的發現及對應儲存格：2.1 節統計出 `sleep_hours` 有 2 筆缺失（學號 6、18），2.2 節找出螢幕時間異常值為學號 9 的 `15.0` 小時。
- 一項資料或推論限制：這份資料的缺失值和 15.0 異常值都是事後人為加上去的模擬數據，只能練習檢查語法，不能當作真實學生的狀況。

**交付前檢核：** 已填完程式空格與註解；已填寫預測、驗證、解釋與延伸；已由上到下執行並確認圖表和數字與文字一致。
